In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ============================================================
# Build a CPLEX LP (MIP) to:
#  - select exactly one management option per stand
#  - maximize landscape-level post-treatment fire resistance
#    (area-weighted score for the target year)
#  - constrain the total treated area to a fixed fraction of
#    the total landscape area
#
# Interpretation:
#   - "before treatment" = current stand state represented in
#     the input options table
#   - "after treatment"  = outcome associated with each option
#     (including Do-nothing), expressed here as score_2027
# ============================================================

# -------------------------
# INPUT / OUTPUT
# -------------------------
INPUT_CSV = Path("management_options_with_post_treatment_scores.csv")
OUTPUT_DIR = Path("model_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_LP = OUTPUT_DIR / "maximize_post_treatment_landscape_score.lp"
OUTPUT_VARMAP = OUTPUT_DIR / "variable_mapping.csv"

# -------------------------

AREA_LIMIT_FRACTION = 0.30  #  ///////Use as you want 
NO_TREATMENT_LABEL = "Do-nothing"
ADD_SYNTHETIC_NO_TREATMENT_IF_MISSING = False

# -------------------------
# HELPER
# -------------------------
def safe_name(name: str) -> str:
    """Make a string safe for CPLEX LP constraint names."""
    return re.sub(r"[^A-Za-z0-9_]", "_", str(name))

# -------------------------
# LOAD INPUT TABLE
# -------------------------
df = pd.read_csv(INPUT_CSV, low_memory=False)

# Required columns in the input table:
#   stand_id    : stand / management unit identifier
#   option_id   : management option identifier
#   treatment   : treatment label
#   score_after  : post-treatment score associated with that option
#   area        : stand area
required_cols = ["stand_id", "option_id", "treatment", "score_2027", "area"]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Available columns: {list(df.columns)}"
    )


# -------------------------
# COMPUTE MODEL COEFFICIENTS
# -------------------------
# Total landscape area counts each stand once
total_area = float(df.groupby("stand_id")["area"].first().sum())
if total_area <= 0:
    raise ValueError("Total landscape area must be greater than zero.")

treated_area_limit = AREA_LIMIT_FRACTION * total_area

# Objective coefficient:
# ensures the objective equals the area-weighted mean post-treatment score
df["objective_coefficient"] = (df["area"] * df["score_2027"]) / total_area

# Treated area coefficient:
# 0 for Do-nothing, area for all active treatments
df["treated_area_coefficient"] = np.where(
    df["treatment"] == NO_TREATMENT_LABEL,
    0.0,
    df["area"]
)

# -------------------------
# ASSIGN BINARY VARIABLES
# -------------------------
df = df.reset_index(drop=True)
df["variable"] = [f"x{i}" for i in range(len(df))]

# Save variable map for traceability
map_columns = [
    "variable", "stand_id", "option_id", "treatment",
    "score_2027", "area", "objective_coefficient", "treated_area_coefficient"
]
df[map_columns].to_csv(OUTPUT_VARMAP, index=False)

print("Number of stands:", df["stand_id"].nunique())
print("Number of options:", len(df))
print("Total landscape area:", total_area)
print("Treated area limit:", treated_area_limit)
print("Variable map written to:", OUTPUT_VARMAP)

# -------------------------
# WRITE CPLEX LP FILE
# -------------------------
lines = []
lines.append("\\ Mixed-integer linear program")
lines.append("\\ Objective: maximize area-weighted post-treatment landscape score")
lines.append(f"\\ Constraint: treated area <= {AREA_LIMIT_FRACTION:.0%} of total landscape area")
lines.append("")
lines.append("Maximize")

obj_terms = [
    f" {row.objective_coefficient:.12g} {row.variable}"
    for row in df.itertuples()
    if pd.notna(row.objective_coefficient) and row.objective_coefficient != 0
]
lines.append(" obj:" + " +".join(obj_terms) if obj_terms else " obj: 0")

lines.append("")
lines.append("Subject To")

# Exactly one option per stand
for stand_id, group in df.groupby("stand_id", sort=False):
    vars_for_stand = group["variable"].tolist()
    if vars_for_stand:
        lines.append(f" c_select_{safe_name(stand_id)}: " + " + ".join(vars_for_stand) + " = 1")

# Total treated area constraint
treated_terms = [
    f" {row.treated_area_coefficient:.12g} {row.variable}"
    for row in df.itertuples()
    if row.treated_area_coefficient != 0
]
lhs = " +".join(treated_terms) if treated_terms else "0"
lines.append(f" c_treated_area_limit: {lhs} <= {treated_area_limit:.12g}")

lines.append("")
lines.append("Binary")
for variable in df["variable"]:
    lines.append(f" {variable}")

lines.append("")
lines.append("End")

OUTPUT_LP.write_text("\n".join(lines), encoding="utf-8")
print("LP file written to:", OUTPUT_LP)


